### Model Training and Analysis

#### Playing around with different ML models to see the best suitability

TF-IDF converts text into numbers so that a machine learning model can understand it.

TF - Term Frequency [Higher the frequency, higher the TF]

IDF - Inverse Document Frequency [Finds the unique word occurences]

Once every attribute is converted into a TF-IDF vector, we compare two vectors.

Instead of comparing the numbers directly, Cosine Similarity measures the angle between them.

Example

Course A

Python
Pandas
NumPy
Machine Learning

Course B

Python
Pandas
Data Analysis

Many common words.

Cosine similarity might be

0.83

Very similar.

Course C

Java
Spring Boot
REST APIs

Similarity with Course A

0.12

Very different.

Why use both together?

TF-IDF creates the numeric representation.

Cosine Similarity compares those representations.

In [36]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [37]:
df = pd.read_csv("../data/processed/personalized_learning_processed.csv")

In [38]:
df.head()

,course_id,name,category,difficulty,estimated_hours,goal_role,prerequisites,skills,concept_overlap_group,module_list,...,target_roles,career_importance,learning_outcomes,practical_application,assessment_method,difficulty_score,career_value_score,skill_growth_score,num_prerequisites,num_skills
0,C001,"Anatomy & Physiology: Regulation, Integration,...",Health,Intermediate,28,Data Scientist,Medical Terminology,"Medical Terminology, Endocrinology, Life Scien...",5,Not Specified,...,Data Scientist,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,3,8,3.65,1,8
1,C002,Excel Essentials and Beyond,Data Science,Intermediate,49,Full Stack,Data Presentation,"Data Presentation, Pivot Tables And Charts, Sp...",64,"Navigate Excel with confidence, leveraging ess...",...,Full Stack,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,3,8,5.67,1,12
2,C003,Project Execution: Running the Project,Business,Beginner,8,AI Engineer,Continuous Improvement Process,"Continuous Improvement Process, Data Storytell...",45,Implement the key quality management concepts ...,...,AI Engineer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,1,6,4.64,1,17
3,C004,Taxation of Multinationals for Everyone,Business,Intermediate,26,ML Engineer,Income Tax,"Income Tax, Tax Planning, Tax, Corporate Tax, ...",74,Participants will gain a comprehensive underst...,...,ML Engineer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,3,8,3.25,1,7
4,C005,Creating Change through Social Entrepreneurship,Business,Intermediate,53,Backend Developer,Advocacy,"Advocacy, Entrepreneurship, Design Thinking, R...",58,Not Specified,...,Backend Developer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,3,8,6.46,1,14


In [39]:
df.tail()

,course_id,name,category,difficulty,estimated_hours,goal_role,prerequisites,skills,concept_overlap_group,module_list,...,target_roles,career_importance,learning_outcomes,practical_application,assessment_method,difficulty_score,career_value_score,skill_growth_score,num_prerequisites,num_skills
295,C296,EKP1: Modernity and the Emergence of Korean Ph...,Arts And Humanities,Beginner,49,AI Engineer,Cultural Diversity,"Cultural Diversity, Ancient History, Research,...",74,Examine the definitions of philosophy and Kore...,...,AI Engineer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,1,6,2.03,1,5
296,C297,Applied Calculus with Python,Data Science,Beginner,34,Full Stack,Applied Mathematics,"Applied Mathematics, Derivatives, Calculus, Gr...",71,Not Specified,...,Full Stack,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,1,6,3.51,1,11
297,C298,Generative AI for Data Science with Copilot,Data Science,Advanced,41,Data Analyst,Responsible AI,"Responsible AI, Data Ethics, Anomaly Detection...",67,Define and differentiate types of generative A...,...,Data Analyst,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,5,10,5.73,1,9
298,C299,Particle Physics: an Introduction,Physical Science And Engineering,Advanced,37,AI Engineer,Mathematical Modeling,"Mathematical Modeling, Experimentation, Resear...",76,Not Specified,...,AI Engineer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,5,10,5.27,1,8
299,C300,Computer Vision in Microsoft Azure,Information Technology,Beginner,19,Backend Developer,Application Development,"Application Development, Data Processing, Mach...",33,How to identify Azure tools and services for c...,...,Backend Developer,Important,Apply None concepts to practical tasks.,Coding Exercises,Coding Assignment,1,6,2.69,1,10


In [40]:
df.shape

(300, 31)

In [41]:
df["combined_features"] = (
    df["skills"].fillna("") + " " +
    df["course_description"].fillna("") + " " +
    df["learning_outcomes"].fillna("") + " " +
    df["core_skills"].fillna("") + " " +
    df["goal_role"].fillna("") + " " +
    df["difficulty"].fillna("") + " " +
    df["category"].fillna("")
)

In [42]:
df["combined_features"].head()

0    Medical Terminology, Endocrinology, Life Scien...
1    Data Presentation, Pivot Tables And Charts, Sp...
2    Continuous Improvement Process, Data Storytell...
3    Income Tax, Tax Planning, Tax, Corporate Tax, ...
4    Advocacy, Entrepreneurship, Design Thinking, R...
Name: combined_features, dtype: str

#### TF - IDF

In [43]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

In [44]:
tfidf_matrix = tfidf.fit_transform(df["combined_features"])

In [45]:
tfidf_matrix.shape

(300, 1067)

#### Cosine Similarity

In [46]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [47]:
cosine_sim.shape

(300, 300)

In [48]:
indices = pd.Series(
    df.index,
    index=df["name"]
).drop_duplicates()

In [49]:
def recommend_courses(course_name, top_n=5):

    # Check if the course exists
    if course_name not in indices:
        print("Course not found.")
        return

    # Get the index of the selected course
    idx = indices[course_name]

    # Get similarity scores for all courses
    similarity_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity (highest first)
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected course itself and keep top recommendations
    similarity_scores = similarity_scores[1:top_n+1]

    # Separate indices and similarity values
    course_indices = [i[0] for i in similarity_scores]
    similarity_values = [i[1] for i in similarity_scores]

    # Get recommended courses
    recommendations = df[
        [
            "name",
            "category",
            "difficulty",
            "goal_role",
            "estimated_hours",
            "skills"
        ]
    ].iloc[course_indices].reset_index(drop=True)

    # Add similarity score as percentage
    recommendations["match_percentage"] = [
        round(score * 100, 2)
        for score in similarity_values
    ]

    return recommendations

In [50]:
df["name"].head(20)

0     Anatomy & Physiology: Regulation, Integration,...
1                           Excel Essentials and Beyond
2                Project Execution: Running the Project
3               Taxation of Multinationals for Everyone
4       Creating Change through Social Entrepreneurship
5        Transforming Exploratory Data Analysis with AI
6                       Introduction to Risk Management
7     Anatomy & Physiology: Filtration and Reproduction
8              Fundamentals of Red Hat Enterprise Linux
9                         Effective Compliance Programs
10                                     IBM COBOL Basics
11                 Low Intermediate English: Technology
12                    Studying at Japanese Universities
13    Introduction to Generative AI for Developers W...
14    EU policy and implementation: making Europe work!
15                  Music Composition: Starting a Piece
16          Decentralized Finance (DeFi) Infrastructure
17    Social Psychology of Attraction and Relati

In [51]:
recommend_courses(
    "Excel Essentials and Beyond"
)

,name,category,difficulty,goal_role,estimated_hours,skills,match_percentage
0,Data Visualization Fundamentals,Data Science,Intermediate,ML Engineer,35,"Statistical Visualization, Histogram, Data Vis...",60.21
1,Harnessing the Power of Data with Power BI,Data Science,Beginner,Data Analyst,48,"Data Visualization Software, Data-Driven Decis...",56.53
2,Problem Solving with Excel,Data Science,Intermediate,Backend Developer,51,"Data Analysis, Microsoft Excel, Data Validatio...",55.79
3,Speed Up Data Analysis and Presentation Building,Data Science,Advanced,ML Engineer,26,"Responsible AI, Data Presentation, Spreadsheet...",51.75
4,Communicating Data Insights with Tableau,Data Science,Intermediate,Backend Developer,47,"Tableau Software, Exploratory Data Analysis, D...",49.29


In [52]:
recommend_courses(
    "Project Execution: Running the Project"
)

,name,category,difficulty,goal_role,estimated_hours,skills,match_percentage
0,Capstone: Applying Project Management in the R...,Business,Advanced,Data Analyst,51,"Requirements Analysis, Stakeholder Management,...",65.18
1,Foundations of Data Science,Data Science,Beginner,ML Engineer,42,"Data Ethics, Workflow Management, Business Ana...",47.58
2,Introduction to Operations Management,Business,Intermediate,AI Engineer,26,"Operational Efficiency, Process Analysis, Cont...",28.40
3,Introduction to Agile Development and Scrum,Computer Science,Beginner,ML Engineer,23,"Sprint Planning, Agile Project Management, Con...",27.86
4,Reputation Crisis? Facebook meets Cambridge An...,Social Sciences,Intermediate,Data Scientist,34,"Business Management, Stakeholder Management, L...",21.55


In [53]:
joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

['../models/tfidf_vectorizer.pkl']

In [54]:
joblib.dump(
    cosine_sim,
    "../models/cosine_similarity.pkl"
)

['../models/cosine_similarity.pkl']

In [55]:
joblib.dump(
    df,
    "../models/course_data.pkl"
)

['../models/course_data.pkl']

In [56]:
import os

os.listdir("../models")

['cosine_similarity.pkl',
 'course_data.pkl',
 'model.pkl',
 'tfidf_vectorizer.pkl']